# Assignment 07: Classification Metrics (100 points)

**Unit**: ML1 Supervised Learning (AI 300)  
**Topics**: Confusion matrix, precision/recall/F1, ROC curve, AUC, precision-recall curve, class imbalance

---

## Background

For a binary classifier with threshold $\tau$: predict positive if score $\geq \tau$.

| | Predicted + | Predicted - |
|---|---|---|
| **Actual +** | TP | FN |
| **Actual -** | FP | TN |

**Precision** $= \frac{TP}{TP + FP}$, **Recall** $= \frac{TP}{TP + FN}$, **F1** $= \frac{2 \cdot P \cdot R}{P + R}$

**ROC curve**: plot TPR vs FPR at all thresholds. **AUC**: area under the ROC curve.

### Notation

| Symbol | Description |
|--------|-------------|
| TPR | True positive rate = recall |
| FPR | False positive rate = FP / (FP + TN) |
| AUC | Area under the ROC curve |

In [ ]:
"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

> **WARNING !!!**
>
> - Beyond importing libraries/modules/classes/functions in the preceding cell, you are **NOT allowed to import anything else for the following purposes**:
>     - **As a part of your final solution.**
>     - **Temporarily import something to assist you to get a solution.**
>
>     **Rule of thumb:** Each part has its particular purpose to intentionally test you something. Do not attempt to find a shortcut to circumvent the rule.

In [ ]:
"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
Simulated classifier output for 8 samples.
"""
scores = np.array([0.95, 0.85, 0.78, 0.66, 0.55, 0.40, 0.30, 0.10])
y_true_8 = np.array([1, 0, 1, 1, 0, 0, 1, 0])

# Larger dataset for ROC/PR curves
n_large = 500
y_true_large = np.random.randint(0, 2, n_large)                  # (500,)
y_scores_large = y_true_large * 0.6 + 0.2 + np.random.randn(n_large) * 0.3
y_scores_large = np.clip(y_scores_large, 0, 1)                  # (500,)

print(f"Small dataset: {len(scores)} samples")
print(f"Large dataset: {n_large} samples, class balance: {y_true_large.mean():.3f}")

---

## Part 1 (20 points, coding task)

**Implement all binary classification metrics.**

Given binary labels `y_true` and `y_pred`, compute: TP, FP, FN, TN, accuracy, precision, recall, F1, specificity, and FPR. Return them in a dictionary.

Also implement the $F_\beta$ score: $F_\beta = \frac{(1+\beta^2) \cdot P \cdot R}{\beta^2 \cdot P + R}$

*Reasoning is not required.*

In [ ]:
def binary_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """
    Compute all binary classification metrics.

    Args:
        y_true: (n,) true binary labels {0, 1}
        y_pred: (n,) predicted binary labels {0, 1}

    Returns:
        dict with keys: tp, fp, fn, tn, accuracy, precision, recall,
                        f1, specificity, fpr
    """
    ### WRITE YOUR SOLUTION HERE ###

    pass


def f_beta_score(precision: float, recall: float, beta: float) -> float:
    """
    F-beta score: (1 + beta^2) * P * R / (beta^2 * P + R)

    Returns 0.0 if both precision and recall are 0.
    """
    ### WRITE YOUR SOLUTION HERE ###

    pass

In [ ]:
"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""
# Test with known values
y_t = np.array([1, 1, 1, 1, 0, 0, 0, 0, 0, 0])
y_p = np.array([1, 1, 0, 0, 0, 0, 0, 0, 1, 0])
m = binary_metrics(y_t, y_p)

assert m['tp'] == 2, f"TP should be 2, got {m['tp']}"
assert m['fp'] == 1, f"FP should be 1, got {m['fp']}"
assert m['fn'] == 2, f"FN should be 2, got {m['fn']}"
assert m['tn'] == 5, f"TN should be 5, got {m['tn']}"
assert abs(m['precision'] - 2/3) < 1e-10
assert abs(m['recall'] - 0.5) < 1e-10
print(f"TP={m['tp']}, FP={m['fp']}, FN={m['fn']}, TN={m['tn']}")
print(f"Precision: {m['precision']:.4f}, Recall: {m['recall']:.4f}, F1: {m['f1']:.4f}")

# F-beta tests
assert abs(f_beta_score(0.8, 0.6, 1.0) - 2*0.8*0.6/(0.8+0.6)) < 1e-10
assert f_beta_score(0.8, 0.6, 2.0) > f_beta_score(0.8, 0.6, 1.0)  # F2 weights recall more
print(f"F1:   {f_beta_score(0.8, 0.6, 1.0):.4f}")
print(f"F2:   {f_beta_score(0.8, 0.6, 2.0):.4f}")
print(f"F0.5: {f_beta_score(0.8, 0.6, 0.5):.4f}")
print("Part 1 passed.")

""" END OF THIS PART """

---

## Part 2 (10 points, coding task)

**Compute the confusion matrix at threshold = 0.5** for the 8-sample dataset.

Print the confusion matrix and all metrics (precision, recall, F1, accuracy).

*Reasoning is not required.*

In [ ]:
### WRITE YOUR SOLUTION HERE ###
# Use scores, y_true_8 from the data cell above
# Threshold at 0.5, compute and print all metrics

pass

""" END OF THIS PART """

---

### The ROC Curve

The **Receiver Operating Characteristic** curve plots TPR (recall) against FPR as the classification threshold varies from high to low. A perfect classifier achieves AUC = 1.0; a random classifier achieves AUC = 0.5.

---

## Part 3 (25 points, coding task)

**Implement ROC curve, precision-recall curve, and AUC computation.**

1. `roc_curve(y_true, y_scores)` returns `(fpr, tpr, thresholds)`. Sort by decreasing score, sweep thresholds.
2. `pr_curve(y_true, y_scores)` returns `(precision, recall, thresholds)`.
3. `auc_trapezoidal(x, y)` computes area under a curve using the trapezoidal rule.
4. Create a 1x2 subplot: left = ROC curve with AUC annotation, right = PR curve with AP annotation. Include the random-classifier baseline on each.

*Reasoning is not required.*

In [ ]:
def roc_curve(y_true: np.ndarray, y_scores: np.ndarray) -> tuple:
    """
    Compute ROC curve.

    Args:
        y_true:   (n,) binary labels
        y_scores: (n,) predicted scores

    Returns:
        fpr:        array of FPR values
        tpr:        array of TPR values
        thresholds: array of thresholds used
    """
    ### WRITE YOUR SOLUTION HERE ###

    pass


def pr_curve(y_true: np.ndarray, y_scores: np.ndarray) -> tuple:
    """
    Compute Precision-Recall curve.

    Returns:
        precision:  array of precision values
        recall:     array of recall values
        thresholds: array of thresholds used
    """
    ### WRITE YOUR SOLUTION HERE ###

    pass


def auc_trapezoidal(x: np.ndarray, y: np.ndarray) -> float:
    """
    Area under curve via trapezoidal rule.

    Args:
        x: (m,) x-coordinates (must be sorted)
        y: (m,) y-coordinates

    Returns:
        Scalar area
    """
    ### WRITE YOUR SOLUTION HERE ###

    pass

In [ ]:
"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""
fpr, tpr, _ = roc_curve(y_true_large, y_scores_large)
our_auc = auc_trapezoidal(fpr, tpr)
print(f"AUC: {our_auc:.4f}")
assert 0.5 < our_auc < 1.0, f"AUC should be between 0.5 and 1.0, got {our_auc}"
assert fpr[0] == 0 and tpr[0] == 0, "ROC curve should start at (0,0)"

prec, rec, _ = pr_curve(y_true_large, y_scores_large)
assert len(prec) == len(rec)
print(f"PR curve points: {len(prec)}")
print("Curve functions validated.")

In [ ]:
### WRITE YOUR SOLUTION HERE ###
# Create 1x2 subplot: ROC curve (left), PR curve (right)
# Include AUC/AP annotations and random baselines

pass

""" END OF THIS PART """

---

## Part 4 (15 points, coding task)

**Threshold selection and class imbalance analysis.**

1. Plot precision, recall, and F1 as functions of threshold (from 0.0 to 1.0). Find and mark the threshold that maximizes F1.
2. Create imbalanced datasets with positive class fractions $\in \{0.5, 0.1, 0.01\}$. For each, train a simple classifier (use: predict score = feature\_0 + noise) and show that **accuracy is misleading** under imbalance while **F1 and AUC** remain informative.

*Reasoning is not required.*

In [ ]:
### WRITE YOUR SOLUTION HERE ###
# Part 1: Plot precision, recall, F1 vs threshold. Mark optimal F1 threshold.
# Part 2: Imbalance analysis for ratios {0.5, 0.1, 0.01}.

pass

""" END OF THIS PART """

---

## Part 5 (15 points, coding task)

**Implement multiclass metrics: macro and micro averaging.**

- **Macro**: compute per-class precision, recall, F1, then average.
- **Micro**: aggregate TP, FP, FN across all classes, then compute global metrics.

*Reasoning is not required.*

In [ ]:
def multiclass_metrics(
    y_true: np.ndarray, y_pred: np.ndarray, average: str = "macro"
) -> dict:
    """
    Compute multiclass precision, recall, F1.

    Args:
        y_true: (n,) integer labels
        y_pred: (n,) predicted integer labels
        average: "macro" or "micro"

    Returns:
        dict with keys: precision, recall, f1
    """
    ### WRITE YOUR SOLUTION HERE ###

    pass

In [ ]:
"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""
y_mc_true = np.array([0, 0, 0, 1, 1, 1, 2, 2, 2])
y_mc_pred = np.array([0, 0, 1, 1, 1, 2, 2, 2, 0])

macro = multiclass_metrics(y_mc_true, y_mc_pred, "macro")
micro = multiclass_metrics(y_mc_true, y_mc_pred, "micro")
print(f"Macro: P={macro['precision']:.4f}, R={macro['recall']:.4f}, F1={macro['f1']:.4f}")
print(f"Micro: P={micro['precision']:.4f}, R={micro['recall']:.4f}, F1={micro['f1']:.4f}")

# Micro precision == micro recall == micro F1 == accuracy (always true)
acc_mc = np.mean(y_mc_true == y_mc_pred)
assert abs(micro['precision'] - acc_mc) < 1e-10, "Micro precision should equal accuracy"
assert abs(micro['recall'] - acc_mc) < 1e-10, "Micro recall should equal accuracy"
print("Part 5 passed.")

""" END OF THIS PART """

---

## Part 6 (15 points, non-coding task)

**Analysis questions.**

1. A medical test has 99% accuracy on a disease that affects 1% of the population. If the test always predicts "healthy", what are the precision, recall, and F1 for the "disease" class? Why is accuracy misleading here?

2. Explain the relationship between the ROC curve and the precision-recall curve. In what scenarios is the PR curve more informative than the ROC curve? (2-3 sentences)

3. Prove that for any classifier, micro-averaged precision, recall, and F1 are all equal (and equal to accuracy) when every sample is assigned exactly one class. (Hint: show $\sum_c TP_c = \sum_c (TP_c + FN_c) = n$ and similarly for $FP$.)

*Reasoning is required.*

### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """